# 05 - Instrumental variables

Instrumental variables can help when treatment is confounded by unobserved factors, provided a valid instrument exists.


## Causal question
Can an instrument recover the causal effect of treatment on outcome when treatment selection is confounded by unobserved factors?

## Causal setup
- Treatment: binary intervention indicator (`treatment`).
- Outcome: simulated response variable (`outcome`).
- Covariates: observed covariate (`x`) used as a control in the first and second stages.
- Unit of analysis: independent observational units in the synthetic IV dataset.

## Estimand
Estimate the ATE of treatment (`treatment`) on outcome (`outcome`).

## Identification assumptions
- Instrument relevance: the instrument is correlated with treatment.
- Exclusion: the instrument only affects outcome through treatment.
- Monotonicity and independence: the instrument is as-if randomly assigned and not related to unobserved outcome shocks after controls.

## Uncertainty and limitations
Report bootstrap uncertainty around both naive and IV point estimates and assess whether the first-stage diagnostics support relevance.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Step execution
This cell performs the next computation. Read the result and tie it back to the causal setup before moving on.


In [ ]:

from causal_inference_lab.data_generators import make_iv_data
from causal_inference_lab.estimators import difference_in_means
from causal_inference_lab.instrumental_variables import instrumental_variables_ate
from causal_inference_lab.uncertainty import bootstrap_ate

dataset = make_iv_data(n=5_000, seed=99)
data = dataset.data

data.head()


## Naive estimate

Treatment is affected by an unobserved confounder, so the naive estimate is biased.


In [ ]:
naive = difference_in_means(data)
naive_bootstrap = bootstrap_ate(
    data=data,
    estimator=lambda frame: difference_in_means(frame),
    n_bootstrap_samples=500,
    seed=42,
    confidence_level=0.95,
    )

print(f"Naive estimate: {naive.estimate:.3f}")
print(
    f"{int(naive_bootstrap.confidence_level*100)}% bootstrap interval: "
    f"[{naive_bootstrap.lower:.3f}, {naive_bootstrap.upper:.3f}]"
)


## Manual two-stage least squares

Stage 1 predicts treatment using the instrument and observed covariates. Stage 2 regresses the outcome on predicted treatment and covariates.


In [ ]:
iv_result = instrumental_variables_ate(data, covariates=["x"])
iv_bootstrap = bootstrap_ate(
    data=data,
    estimator=lambda frame: instrumental_variables_ate(frame, covariates=["x"]).effect,
    n_bootstrap_samples=500,
    seed=42,
    confidence_level=0.95,
)

print(f"First-stage F-stat:      {iv_result.first_stage_f_stat:.2f}")
print(f"First-stage R2:          {iv_result.first_stage_r2:.3f}")
print(f"Instrument coefficient:  {iv_result.instrument_coefficient:.3f}")
print(f"Instrument is weak:      {iv_result.instrument_is_weak}")
print(f"IV estimate:            {iv_result.effect.estimate:.3f}")
print(f"True effect:             {dataset.true_ate:.3f}")
print(
    f"{int(iv_bootstrap.confidence_level*100)}% bootstrap interval: "
    f"[{iv_bootstrap.lower:.3f}, {iv_bootstrap.upper:.3f}]"
)


**Interpretation.** The naive estimate is expected to be inconsistent when unobserved confounding is present. IV should move closer to the true effect only when instrument relevance and exclusion hold, and the first-stage diagnostics provide support for relevance.
